# 情绪风格 LoRA 插值实验

本 notebook 演示 LoRA 参数空间插值的核心流程：
1. 加载基座模型和多个风格 LoRA
2. 进行权重插值
3. 对比不同权重组合的生成结果

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.interpolate import InterpolatableLoRA, interpolate_lora_dicts
from src.generate import generate_with_interpolation
from src.utils import STYLE_NAMES

print("Imports successful.")

In [ ]:
# 配置
MODEL_NAME = "./models/Qwen/Qwen3-1.7B"
LORA_DIR = "../outputs/lora"

# 加载模型
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto",
)
print("Model loaded.")

In [ ]:
# 加载 LoRA 适配器
import os
lora_paths = {}
for style in ["empathetic", "rational", "encouraging", "calm_safe"]:
    p = os.path.join(LORA_DIR, style)
    if os.path.exists(p):
        lora_paths[style] = p

print(f"Found {len(lora_paths)} LoRA adapters: {list(lora_paths.keys())}")

if lora_paths:
    wrapper = InterpolatableLoRA(model, lora_paths, interpolation_mode="weight")
else:
    print("No LoRA adapters found. Using base model for demonstration only.")
    wrapper = None

In [ ]:
# 实验：不同权重组合对比
test_input = "最近工作压力很大，感觉喘不过气来"

weight_configs = [
    {"empathetic": 1.0, "rational": 0.0, "encouraging": 0.0, "calm_safe": 0.0},
    {"empathetic": 0.0, "rational": 1.0, "encouraging": 0.0, "calm_safe": 0.0},
    {"empathetic": 0.0, "rational": 0.0, "encouraging": 1.0, "calm_safe": 0.0},
    {"empathetic": 0.0, "rational": 0.0, "encouraging": 0.0, "calm_safe": 1.0},
    {"empathetic": 0.5, "rational": 0.5, "encouraging": 0.0, "calm_safe": 0.0},  # 共情+理性
    {"empathetic": 0.3, "rational": 0.1, "encouraging": 0.5, "calm_safe": 0.1},  # 以鼓励为主
]

for i, weights in enumerate(weight_configs):
    label = " + ".join(f"{STYLE_NAMES[k]}:{v}" for k, v in weights.items() if v > 0)
    print(f"[{i+1}] {label}")
    if wrapper is not None:
        response, _ = generate_with_interpolation(
            wrapper, tokenizer, test_input, weights,
            max_new_tokens=256, temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
        )
        print(f"    {response}")
    else:
        print(f"    (需要先训练 LoRA 适配器)")
    print()

## 可视化：风格空间中的连续过渡

下面展示在 共情-理性 二维平面上，不同点对应的生成结果。

In [ ]:
# 在 empathetic-rational 轴上进行连续采样
alphas = np.linspace(0, 1, 5)
print("共情 → 理性 的连续过渡：")
print("=" * 60)

for alpha in alphas:
    weights = {
        "empathetic": 1 - alpha,
        "rational": alpha,
        "encouraging": 0.0,
        "calm_safe": 0.0,
    }
    e_pct = (1 - alpha) * 100
    r_pct = alpha * 100
    print(f"共情 {e_pct:.0f}% / 理性 {r_pct:.0f}%:")
    
    if wrapper is not None:
        response, _ = generate_with_interpolation(
            wrapper, tokenizer, test_input, weights,
            max_new_tokens=128, temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
        )
        print(f"  {response[:200]}")
    else:
        print(f"  (需要先训练 LoRA 适配器)")
    print()